# 02 — Detekcija sistematskih (logičkih) grešaka

Nakon inicijalnog pregleda, u ovoj svesci proveravamo **logičke
nekonzistentnosti** između kolona — kombinacije vrednosti koje nisu
statistički čudne, ali su logički nemoguće ili sumnjive.

Ovakve greške ne mogu se otkriti standardnim proverama (`isnull()`,
`describe()`, box-plot) jer je svaka vrednost pojedinačno u redu — problem
nastaje tek kada se pogledaju **odnosi između kolona**.

**Napomena:** ova sekcija ide **pre podele na trening i test skup** jer se
radi o strukturnim (objektivnim) proverama koje ne zavise od statistike
podataka. Nemoguća kombinacija ostaje nemoguća bez obzira na distribuciju.

## 1. Učitavanje podataka

Učitavamo isti skup podataka kao u prethodnoj svesci i vršimo istu konverziju
kolone `TotalCharges` u numerički tip. Ovim počinjemo sa istim stanjem
podataka kao na kraju sveske `01`.

In [1]:
import pandas as pd
import numpy as np

#Podesavanje pandas-a: prikazuj sve kolone bez skracivanja
pd.set_option("display.max_columns", None)

#Ucitavamo CSV fajl u DataFrame df.
df = pd.read_csv("../WA_Fn-UseC_-Telco-Customer-Churn.csv")
# Konvertujemo TotalCharges u numerički tip (isto kao u svesci 01).
# Ovo je strukturna popravka koju smo dogovorili kao izuzetak od pravila
# "ništa se ne menja pre split-a" — jer je čisto pitanje tipa podataka,
# ne odluka bazirana na statistici.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(f"Podaci ucitani: {df.shape[0]} redova x {df.shape[1]} kolona")

Podaci ucitani: 7043 redova x 21 kolona


## 2. Konzistentnost: PhoneService ↔ MultipleLines

**Logička provera:** ako korisnik nema telefonsku uslugu (`PhoneService = No`),
onda `MultipleLines` **mora biti** `"No phone service"`. Bilo koja druga
vrednost (`"Yes"` ili `"No"`) bila bi logički nemoguća — kako korisnik može
imati (ili ne imati) više telefonskih linija ako uopšte nema telefonsku uslugu?

Proveravamo da li ova konzistentnost važi za sve redove u skupu.

In [2]:
# Izdvajamo redove gde korisnik nema telefonsku uslugu.
# df["PhoneService"] == "No" vraća seriju True/False za svaki red.
bez_telefona = df[df["PhoneService"] == "No"]

print(f"Broj korisnika bez telefonske usluge: {len(bez_telefona)}")

# Za te korisnike, prikazujemo koje vrednosti imaju u MultipleLines.
# Očekivanje: sve treba da bude "No phone service".
print("\nVrednosti MultipleLines za korisnike bez telefonske usluge:")
print(bez_telefona["MultipleLines"].value_counts())

Broj korisnika bez telefonske usluge: 682

Vrednosti MultipleLines za korisnike bez telefonske usluge:
MultipleLines
No phone service    682
Name: count, dtype: int64


### Rezultat provere
Svi korisnici bez telefonske usluge (ukupno 682) imaju vrednost
`"No phone service"` u koloni `MultipleLines`. Konzistentnost je
**potvrđena** — nema logičkih grešaka između ovih dveju kolona.

## 3. Konzistentnost: InternetService ↔ dodatne internet usluge

**Logička provera:** ako korisnik nema internet uslugu (`InternetService = No`),
onda sve **dodatne internet usluge** moraju biti `"No internet service"`.
Bilo koja druga vrednost (`"Yes"` ili `"No"`) bila bi logički nemoguća —
kako korisnik može imati online zaštitu, backup ili streaming ako uopšte
nema internet?

Dodatne internet usluge su:
- `OnlineSecurity`
- `OnlineBackup`
- `DeviceProtection`
- `TechSupport`
- `StreamingTV`
- `StreamingMovies`

Ovaj put proveravamo konzistentnost **preko 6 kolona odjednom**.

In [3]:
# Izdvajamo redove gde korisnik nema internet uslugu.
bez_interneta = df[df["InternetService"] == "No"]

print(f"Broj korisnika bez internet usluge: {len(bez_interneta)}")

# Lista dodatnih internet usluga koje proveravamo.
dodatne_usluge = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

# Za svaku od tih kolona prikazujemo raspodelu vrednosti kod korisnika bez interneta.
# Očekivanje: sve treba da bude "No internet service".
print("\nRaspodela vrednosti za korisnike bez interneta:")
for usluga in dodatne_usluge:
    print(f"\n{usluga}:")
    print(bez_interneta[usluga].value_counts())

Broj korisnika bez internet usluge: 1526

Raspodela vrednosti za korisnike bez interneta:

OnlineSecurity:
OnlineSecurity
No internet service    1526
Name: count, dtype: int64

OnlineBackup:
OnlineBackup
No internet service    1526
Name: count, dtype: int64

DeviceProtection:
DeviceProtection
No internet service    1526
Name: count, dtype: int64

TechSupport:
TechSupport
No internet service    1526
Name: count, dtype: int64

StreamingTV:
StreamingTV
No internet service    1526
Name: count, dtype: int64

StreamingMovies:
StreamingMovies
No internet service    1526
Name: count, dtype: int64


### Rezultat provere

Svi korisnici bez internet usluge (ukupno 1526) imaju vrednost
`"No internet service"` u svih **6 dodatnih internet usluga**
(`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
`StreamingTV`, `StreamingMovies`). Konzistentnost je **potvrđena** — nema
logičkih grešaka u vezi internet usluga.

## 4. Efikasnija provera pomoću vektorizovanih operacija

Provera iz prethodne sekcije radi, ali ispisuje 6 tabela. Za brzo pregledanje,
efikasniji način je da direktno prebrojimo **broj problematičnih redova** —
onih koji krše logičku konzistentnost.

Koristimo pandas metod `.isin()` koji za svaki red vraća `True` ako je
vrednost u zadatoj listi.

In [4]:
# Za svaku dodatnu uslugu proveravamo koliko korisnika BEZ interneta ima
# vrednost različitu od "No internet service" (što bi bila logička greška).

print("Broj problematičnih redova po koloni:")


for usluga in dodatne_usluge:
    # bez_interneta[usluga] != "No internet service" vraća True za problematične redove
    broj_problema = (bez_interneta[usluga] != "No internet service").sum()
    print(f"  {usluga}: {broj_problema}")

Broj problematičnih redova po koloni:
  OnlineSecurity: 0
  OnlineBackup: 0
  DeviceProtection: 0
  TechSupport: 0
  StreamingTV: 0
  StreamingMovies: 0


## 5. Provera korisnika sa `tenure = 0`

Kolona `tenure` predstavlja broj meseci koliko je korisnik kod kompanije.
Vrednost `0` označava korisnike koji su tek započeli korišćenje usluge.

Ova provera je važna jer:
- Pomaže da razumemo koliko takvih „novih" korisnika ima u skupu
- Već znamo (iz sveske `01`) da su to isti korisnici kod kojih fali `TotalCharges`
- Vredi eksplicitno prijaviti ovu činjenicu radi transparentnosti

In [5]:
# Broj korisnika sa tenure = 0.
tenure_nula = df[df["tenure"] == 0]
print(f"Broj korisnika sa tenure = 0: {len(tenure_nula)}")

# Provera: da li se to poklapa sa brojem korisnika kojima fali TotalCharges?
totalcharges_nan = df[df["TotalCharges"].isnull()]
print(f"Broj korisnika sa TotalCharges = NaN: {len(totalcharges_nan)}")

# Provera identiteta: da li su to iste osobe?
# Uzimamo customerID iz obe grupe i poredimo.
# print(set(tenure_nula["customerID"]))
# print(set(totalcharges_nan["customerID"]))
isti_korisnici = set(tenure_nula["customerID"]) == set(totalcharges_nan["customerID"])
print(f"Da li su to isti korisnici? {isti_korisnici}")

Broj korisnika sa tenure = 0: 11
Broj korisnika sa TotalCharges = NaN: 11
Da li su to isti korisnici? True


### Rezultat provere

Skup sadrži **11 korisnika sa `tenure = 0`**. Poređenje `customerID` vrednosti
potvrđuje da su to **iste osobe** čiji `TotalCharges` je nedostajao

Ovi korisnici su tehnički validni (tek započeta pretplata), ali će zbog
nedostajuće vrednosti `TotalCharges` biti predmet posebnog tretmana u
narednim sveskama (nakon podele na trening/test skup).

---

## 6. Zaključak

Kroz proveru sistematskih grešaka utvrdili smo da je skup podataka
**logički konzistentan**:

**Provere koje su prošle bez grešaka:**
1. Svi korisnici bez telefonske usluge imaju `MultipleLines = "No phone service"`
2. Svi korisnici bez interneta imaju `"No internet service"` u svih 6 dodatnih
   internet usluga
3. Korisnici sa `tenure = 0` odgovaraju istoj grupi kod koje fali `TotalCharges`

**Šta ovo znači za dalji rad:**
- Kolone `MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`,
  `TechSupport`, `StreamingTV`, `StreamingMovies` sadrže konzistentnu informaciju
  o odsustvu usluge (`"No phone service"` / `"No internet service"`)
- To otvara mogućnost feature engineering-a: možemo razmišljati o spajanju
  vrednosti (npr. `"No"` i `"No internet service"` u istu kategoriju) ili
  o pravljenju novih izvedenih atributa


**Sledeći korak:** sveska `03_train_test_split` — podela podataka na trening
i test skup pre bilo kakve analize ili obrade koja zavisi od
statistike podataka.